In [2]:
from dotenv import load_dotenv
load_dotenv()

ModuleNotFoundError: No module named 'dotenv'

In [13]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [27]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    strike_rate: float
    boundary_percent: float
    balls_per_boundary: float
    summary: str

In [41]:
# function to calculate strike rate

def calculate_strike_rate(state: BatsmanState):
    runs = state['runs']
    balls = state['balls']

    strike_rate = round((runs/balls)*100, 2)

    return {"strike_rate": strike_rate}  # partial update


In [38]:
# function to calculate boundary percentage

def calculate_boundary_percentage(state: BatsmanState):
    runs = state['runs']
    fours = state['fours']
    sixes = state['sixes']

    boundary_runs = fours*4 + sixes*6

    boundary_percent = round((boundary_runs/runs)*100, 2)

    return {"boundary_percent": boundary_percent}  # partial update



In [39]:
# function to calculate balls per boundary

def calculate_balls_per_boundary(state: BatsmanState):
    fours = state['fours']
    sixes = state['sixes']
    balls = state['balls']

    total_boundaries = fours + sixes
    balls_per_boundary = round(balls/total_boundaries, 2)

    return {"balls_per_boundary": balls_per_boundary}  # partial update


In [40]:
def create_summary(state: BatsmanState):
    state['summary'] = f"""
strike rate = {state['strike_rate']} \n
boundary_percent = {state['boundary_percent']} \n
balls_per_boundary = {state['balls_per_boundary']} \n
"""

    return state

In [42]:
# graph creation
graph = StateGraph(BatsmanState)

# add node to the graph
graph.add_node('calculate_strike_rate', calculate_strike_rate)
graph.add_node('calculate_boundary_percentage', calculate_boundary_percentage)
graph.add_node('calculate_balls_per_boundary', calculate_balls_per_boundary)
graph.add_node('create_summary', create_summary)

# add edges to the graph
graph.add_edge(START, 'calculate_strike_rate')
graph.add_edge(START, 'calculate_boundary_percentage')
graph.add_edge(START, 'calculate_balls_per_boundary')

graph.add_edge('calculate_strike_rate', 'create_summary')
graph.add_edge('calculate_boundary_percentage', 'create_summary')
graph.add_edge('calculate_balls_per_boundary', 'create_summary')

graph.add_edge('create_summary', END)

# compile the graph
workflow = graph.compile()

In [43]:
# workflow execution
initial_state = {'runs': 87, 'balls': 78, 'fours': 8, 'sixes': 4}

final_state = workflow.invoke(initial_state)

print(final_state)
print(final_state['summary'])

{'runs': 87, 'balls': 78, 'fours': 8, 'sixes': 4, 'strike_rate': 111.54, 'boundary_percent': 64.37, 'balls_per_boundary': 6.5, 'summary': '\nstrike rate = 111.54 \n\nboundary_percent = 64.37 \n\nballs_per_boundary = 6.5 \n\n'}

strike rate = 111.54 

boundary_percent = 64.37 

balls_per_boundary = 6.5 


